#  Airline On-Time Performance — Combined EDA (2020–2025)





# Part 1 — Route Reliability & Distance/Time Patterns


This part builds a working dataframe (`df1`) of the key operational columns, sanity-checks it for
duplicate flight keys and implausible minimum values, then digs into **route-level reliability**,
**delay by trip distance**, **directional imbalance between airport pairs**, and a **airline ×
month seasonal delay table**.


### 1.1 Environment setup & load data

In [1]:
import pyspark

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,Current session?
0,application_1784624419766_0001,pyspark3,idle,Link,Link,✔


SparkSession available as 'spark'.


In [4]:
df = spark.read.parquet('s3://airline-dataset-2020-2025/Silver/')

### 1.2 Select the analysis columns
Trim the raw table down to the 34 fields actually needed for the EDA below (dates, carrier/route
identifiers, delay components, and operational metrics).

In [5]:
# Select required columns
df1 = df.select(
    'FlightDate',
    'Year',
    'Month',
    'Quarter',
    'DayOfMonth',
    'DayOfWeek',
    'Marketing_Airline_Network',
    'Flight_Number_Marketing_Airline',
    'Operating_Airline',
    'Operated_or_Branded_Code_Share_Partners',
    'Origin',
    'OriginState',
    'OriginStateName',
    'OriginCityName',
    'Dest',
    'DestState',
    'DestStateName',
    'DestCityName',
    'CRSDepTime',
    'CRSArrTime',
    'ArrDelay',
    'ArrDel15',
    'DepDelay',
    'DepDel15',
    'CarrierDelay',
    'WeatherDelay',
    'NASDelay',
    'SecurityDelay',
    'LateAircraftDelay',
    'AirTime',
    'Cancelled',
    'Diverted',
    'Distance',
    'TaxiOut',
    'TaxiIn'
)

### 1.3 Schema check

In [10]:
df1.printSchema()

root
 |-- FlightDate: timestamp (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Quarter: integer (nullable = true)
 |-- DayOfMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- Flight_Number_Marketing_Airline: integer (nullable = true)
 |-- Operating_Airline: string (nullable = true)
 |-- Operated_or_Branded_Code_Share_Partners: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginState: string (nullable = true)
 |-- OriginStateName: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestState: string (nullable = true)
 |-- DestStateName: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- CRSArrTime: integer (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-

### 1.4 Data quality check — duplicate flight keys
A flight should be uniquely identified by its date, marketing carrier, flight number, origin,
destination, and scheduled departure time. Checking for duplicates on this candidate key verifies
there's no accidental double-counting in the source data.

In [6]:
candidate_key = [
    "FlightDate",
    "Marketing_Airline_Network",
    "Flight_Number_Marketing_Airline",
    "Origin",
    "Dest",
    "CRSDepTime",
]

In [10]:
from pyspark.sql import functions as F
duplicate_keys = (
    df1.groupBy(candidate_key)
    .count()
    .filter(F.col("count") > 1)
    .count()
)
print("total duplicate",duplicate_keys)

total duplicate 0

### 1.5 minimum values per numeric column
A quick scan of the minimum value in every numeric column, useful for spotting negative values
that shouldn't exist (e.g. negative distances) or sentinel/placeholder values.

In [15]:
from pyspark.sql.types import *

numeric_types = (
    ByteType, ShortType, IntegerType,
    LongType, FloatType, DoubleType, DecimalType
)

for field in df1.schema.fields:
    if isinstance(field.dataType, numeric_types):
        minimum = df1.select(F.min(field.name)).first()[0]
        print(f"{field.name} : {minimum}")

Year : 2020
Month : 1
Quarter : 1
DayOfMonth : 1
DayOfWeek : 1
Flight_Number_Marketing_Airline : 1
CRSDepTime : 1
CRSArrTime : 1
ArrDelay : -139.0
ArrDel15 : 0.0
DepDelay : -131.0
DepDel15 : 0.0
CarrierDelay : 0.0
WeatherDelay : 0.0
NASDelay : 0.0
SecurityDelay : 0.0
LateAircraftDelay : 0.0
AirTime : 5.0
Cancelled : 0.0
Diverted : 0.0
Distance : 11.0
TaxiOut : 0.0
TaxiIn : 0.0

### 1.6 Null-value audit
Counts how many nulls appear in each selected column — important context before averaging delay
columns, since nulls typically correspond to cancelled/diverted flights that never accrued a delay
value.

In [9]:
from pyspark.sql.functions import col, sum, when

# Count NULL values
null_counts = df1.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df1.columns
])

null_counts.show(vertical=True,truncate=False)

-RECORD 0-------------------------------------------
 FlightDate                              | 0        
 Year                                    | 0        
 Month                                   | 0        
 Quarter                                 | 0        
 DayOfMonth                              | 0        
 DayOfWeek                               | 0        
 Marketing_Airline_Network               | 0        
 Flight_Number_Marketing_Airline         | 1        
 Operating_Airline                       | 0        
 Operated_or_Branded_Code_Share_Partners | 0        
 Origin                                  | 0        
 OriginState                             | 0        
 OriginStateName                         | 0        
 OriginCityName                          | 0        
 Dest                                    | 0        
 DestState                               | 0        
 DestStateName                           | 0        
 DestCityName                            | 0  

### 1.7 Cardinality of categorical columns
Distinct-value counts for every categorical column (airlines, airports, states, cities) — useful
for knowing how much a `groupBy`/pivot on each column will fan out.

In [21]:
from pyspark.sql.functions import countDistinct

categorical_cols = [
    "Marketing_Airline_Network",
    "Operating_Airline",
    "Operated_or_Branded_Code_Share_Partners",
    "Origin",
    "OriginState",
    "OriginStateName",
    "OriginCityName",
    "Dest",
    "DestState",
    "DestStateName",
    "DestCityName"
]

unique_counts = df.select([
    countDistinct(c).alias(c)
    for c in categorical_cols
])

unique_counts.show(vertical=True,truncate=False)

-RECORD 0--------------------------------------
 Marketing_Airline_Network               | 10  
 Operating_Airline                       | 25  
 Operated_or_Branded_Code_Share_Partners | 15  
 Origin                                  | 390 
 OriginState                             | 53  
 OriginStateName                         | 53  
 OriginCityName                          | 384 
 Dest                                    | 391 
 DestState                               | 53  
 DestStateName                           | 53  
 DestCityName                            | 385

---
# Part 2 — Route Reliablity, Completion Rates & Geographic Patterns

This part starts from scratch with its own Spark session: loads the same Silver dataset, profiles
it (nulls, schema, cardinality), and then works through completion/cancellation rates, state- and
city-level traffic and delay patterns, year-over-year trends, and summary statistics for every
delay-related numeric column.


### 2.1 Route reliability score
For every Origin–Destination pair with at least 500 flights, compute average arrival delay,
% of flights delayed 15+ minutes, and % cancelled — then combine all three (min-max normalized)
into a single **`UnreliabilityScore`** so routes can be ranked on one composite metric instead of
three separate ones.

In [16]:
route_reliability = (
    df1
    .groupBy("Origin", "Dest")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum(F.when(F.col("ArrDel15") == 1, 1).otherwise(0)) / F.count("*") * 100, 2).alias("DelayedPct"),
        F.round(F.sum("Cancelled") / F.count("*") * 100, 2).alias("CancelPct"),
        F.round(F.avg("Distance"), 0).alias("AvgDistance"),
    )
    .filter(F.col("Flights") >= 500)  # statistically meaningful routes only
)

In [17]:
# Min-max normalize the three "badness" signals and combine into one score (higher = less reliable)
stats = route_reliability.select(
    F.min("DelayedPct").alias("min_d"), F.max("DelayedPct").alias("max_d"),
    F.min("CancelPct").alias("min_c"), F.max("CancelPct").alias("max_c"),
    F.min("AvgArrDelay").alias("min_a"), F.max("AvgArrDelay").alias("max_a"),
).collect()[0]

route_reliability = (
    route_reliability
    .withColumn("norm_delay", (F.col("DelayedPct") - stats["min_d"]) / (stats["max_d"] - stats["min_d"]))
    .withColumn("norm_cancel", (F.col("CancelPct") - stats["min_c"]) / (stats["max_c"] - stats["min_c"]))
    .withColumn("norm_avgdelay", (F.col("AvgArrDelay") - stats["min_a"]) / (stats["max_a"] - stats["min_a"]))
    .withColumn("UnreliabilityScore", F.round((F.col("norm_delay") + F.col("norm_cancel") + F.col("norm_avgdelay")) / 3, 4))
    .drop("norm_delay", "norm_cancel", "norm_avgdelay")
)

print("Least reliable high-volume routes:")
route_reliability.orderBy(F.desc("UnreliabilityScore")).show(15, truncate=False)

print("Most reliable high-volume routes:")
route_reliability.orderBy("UnreliabilityScore").show(15, truncate=False)

Least reliable high-volume routes:
+------+----+-------+-----------+----------+---------+-----------+------------------+
|Origin|Dest|Flights|AvgArrDelay|DelayedPct|CancelPct|AvgDistance|UnreliabilityScore|
+------+----+-------+-----------+----------+---------+-----------+------------------+
|ASE   |AUS |635    |34.45      |38.11     |8.82     |812.0      |0.7588            |
|RNO   |JFK |931    |59.1       |40.06     |4.62     |2411.0     |0.7419            |
|LEX   |FLL |763    |40.26      |40.5      |6.29     |865.0      |0.7172            |
|USA   |FLL |1611   |32.08      |34.7      |8.38     |643.0      |0.709             |
|PBG   |FLL |809    |23.12      |33.5      |8.9      |1334.0     |0.678             |
|ASE   |ATL |1343   |34.72      |35.82     |6.85     |1304.0     |0.6777            |
|TVC   |EWR |522    |28.8       |28.35     |9.2      |644.0      |0.6762            |
|GSP   |FLL |541    |35.68      |39.0      |5.73     |620.0      |0.6674            |
|IAG   |SFB |527   

### 2.2 Delay by distance band
Flights are bucketed into short-haul (<500 mi), medium-haul (500–1,500 mi), and long-haul
(1,500+ mi) to see whether delay severity scales with trip length — both in raw minutes and
normalized per 100 miles flown.

In [18]:
distance_band_delay = (
    df1
    .withColumn(
        "DistanceBand",
        F.when(F.col("Distance") < 500, "Short-haul (<500mi)")
         .when(F.col("Distance") < 1500, "Medium-haul (500-1500mi)")
         .otherwise("Long-haul (1500mi+)")
    )
    .groupBy("DistanceBand")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("ArrDelay") / F.avg("Distance") * 100, 4).alias("DelayPer100Miles"),
        F.round(F.sum(F.when(F.col("ArrDel15") == 1, 1).otherwise(0)) / F.count("*") * 100, 2).alias("DelayedPct"),
    )
)
distance_band_delay.show(truncate=False)


+------------------------+--------+-----------+----------------+----------+
|DistanceBand            |Flights |AvgArrDelay|DelayPer100Miles|DelayedPct|
+------------------------+--------+-----------+----------------+----------+
|Medium-haul (500-1500mi)|20867305|6.08       |0.7007          |19.59     |
|Long-haul (1500mi+)     |4791925 |3.9        |0.1893          |19.63     |
|Short-haul (<500mi)     |15251023|4.52       |1.4972          |17.16     |
+------------------------+--------+-----------+----------------+----------+

### 2.3 Directional route comparison (A→B vs. B→A)
For airport pairs served in both directions, this compares the average delay flying A→B against
B→A to surface asymmetric bottlenecks (e.g. congestion or prevailing winds that hurt one direction
more than the other).

In [19]:
from pyspark.sql import functions as F

route_pairs = route_reliability.select(
    "Origin", "Dest", "AvgArrDelay", "DelayedPct", "Flights"
)

a_to_b = route_pairs.alias("a")
b_to_a = route_pairs.alias("b")

directional_compare = (
    a_to_b.join(
        b_to_a,
        (F.col("a.Origin") == F.col("b.Dest")) &
        (F.col("a.Dest") == F.col("b.Origin"))
    )
    .filter(F.col("a.Origin") < F.col("a.Dest"))  # Avoid duplicate pairs
    .select(
        F.col("a.Origin").alias("AirportA"),
        F.col("a.Dest").alias("AirportB"),
        F.round(F.col("a.AvgArrDelay"), 2).alias("A_to_B_AvgDelay"),
        F.round(F.col("b.AvgArrDelay"), 2).alias("B_to_A_AvgDelay"),

        # Absolute difference in delays
        F.round(
            F.abs(F.col("a.AvgArrDelay") - F.col("b.AvgArrDelay")),
            2
        ).alias("DelayGap"),

        # Average delay across both directions
        F.round(
            (F.col("a.AvgArrDelay") + F.col("b.AvgArrDelay")) / 2,
            2
        ).alias("AvgDelayGap")
    )
    .orderBy(F.desc("DelayGap"))
)

directional_compare.show(20, truncate=False)

+--------+--------+---------------+---------------+--------+-----------+
|AirportA|AirportB|A_to_B_AvgDelay|B_to_A_AvgDelay|DelayGap|AvgDelayGap|
+--------+--------+---------------+---------------+--------+-----------+
|PVU     |SNA     |5.19           |49.66          |44.47   |27.42      |
|JFK     |RNO     |19.95          |59.1           |39.15   |39.53      |
|AUS     |HNL     |45.46          |6.58           |38.88   |26.02      |
|ALB     |DFW     |11.22          |44.79          |33.57   |28.01      |
|IDA     |LAS     |36.67          |4.63           |32.04   |20.65      |
|AZA     |IDA     |3.43           |32.79          |29.36   |18.11      |
|DFW     |HDN     |6.71           |34.4           |27.69   |20.56      |
|CVG     |JAX     |13.77          |41.04          |27.27   |27.41      |
|HHH     |LGA     |17.94          |-8.72          |26.66   |4.61       |
|LAS     |MFR     |5.75           |31.78          |26.03   |18.77      |
|ORD     |SUN     |6.62           |32.6           |

### 2.4 Airline × Month delay heatmap (table form)
Average arrival delay per airline, pivoted so each month is its own column — a compact way to spot
which carriers struggle most in peak summer/holiday months.

In [20]:
airline_month = (
    df1.groupBy("Marketing_Airline_Network", "Month")
    .agg(F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"))
)

airline_month_pivot = (
    airline_month
    .groupBy("Marketing_Airline_Network")
    .pivot("Month")
    .agg(F.first("AvgArrDelay"))
    .orderBy("Marketing_Airline_Network")
)
print("Average Arrival Delay (min): Airline x Month")
airline_month_pivot.show(truncate=False)

Average Arrival Delay (min): Airline x Month
+-------------------------+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Marketing_Airline_Network|1   |2    |3    |4    |5    |6    |7    |8    |9    |10   |11   |12   |
+-------------------------+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|AA                       |5.55|4.96 |4.46 |4.69 |8.89 |14.51|15.72|10.5 |2.74 |3.16 |2.35 |7.49 |
|AS                       |3.0 |1.87 |1.08 |0.93 |1.49 |4.64 |3.91 |3.59 |0.34 |-0.02|1.17 |6.37 |
|B6                       |8.28|9.75 |9.02 |11.08|8.14 |19.11|24.79|17.26|10.48|5.21 |5.72 |17.08|
|DL                       |1.86|-0.34|-0.35|0.96 |2.31 |6.03 |8.72 |2.5  |-2.3 |-2.59|-2.08|4.38 |
|F9                       |9.19|7.33 |12.85|15.42|15.11|20.95|21.13|15.72|7.37 |8.5  |7.53 |12.71|
|G4                       |9.69|9.84 |12.74|11.38|10.84|18.02|21.36|13.7 |6.51 |9.38 |8.14 |15.53|
|HA                       |5.15|3.77 |4.68 |6.21 |6.3  |4.57 |5.

### 2.5 Flight completion rate

In [14]:
from pyspark.sql import functions as F

total_flights = df1.count()

completion_rate = (
    df1.agg(
        F.sum("Cancelled").alias("CancelledFlights")
    )
    .withColumn("CompletedFlights", F.lit(total_flights) - F.col("CancelledFlights"))
    .withColumn(
        "FlightCompletionRate(%)",
        F.round(F.col("CompletedFlights") / total_flights * 100, 2)
    )
)

completion_rate.show(truncate=False)

+----------------+----------------+-----------------------+
|CancelledFlights|CompletedFlights|FlightCompletionRate(%)|
+----------------+----------------+-----------------------+
|917084.0        |3.9993169E7     |97.76                  |
+----------------+----------------+-----------------------+

### 2.6 Marketing vs. operating flight volume gap

Compares how many flights each carrier code is associated with as the *marketing* airline vs. the *operating* airline — a large gap usually indicates heavy codeshare/regional-partner activity (e.g. a major carrier selling seats on flights physically operated by a regional partner).

In [15]:
marketing = (
    df1.groupBy("Marketing_Airline_Network")
    .count()
    .withColumnRenamed("count", "MarketingFlights")
)

operating = (
    df1.groupBy("Operating_Airline")
    .count()
    .withColumnRenamed("Operating_Airline", "Marketing_Airline_Network")
    .withColumnRenamed("count", "OperatingFlights")
)

marketing_operating_gap = (
    marketing.join(operating, "Marketing_Airline_Network", "outer")
    .fillna(0)
    .withColumn(
        "FlightVolumeGap",
        F.abs(F.col("MarketingFlights") - F.col("OperatingFlights"))
    )
    .orderBy(F.desc("FlightVolumeGap"))
)

marketing_operating_gap.show(truncate=False)

+-------------------------+----------------+----------------+---------------+
|Marketing_Airline_Network|MarketingFlights|OperatingFlights|FlightVolumeGap|
+-------------------------+----------------+----------------+---------------+
|AA                       |10407897        |5078840         |5329057        |
|OO                       |0               |4343633         |4343633        |
|UA                       |7427635         |3669461         |3758174        |
|DL                       |8526129         |5242763         |3283366        |
|YX                       |0               |1812974         |1812974        |
|MQ                       |0               |1521850         |1521850        |
|9E                       |0               |1354202         |1354202        |
|OH                       |0               |1300460         |1300460        |
|AS                       |2238407         |1292316         |946091         |
|YV                       |0               |670438          |670

### 2.7 Average arrival delay by state / city

In [23]:
origin_state_delay = (
    df1.groupBy("OriginStateName","OriginCityName")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay")
    )
    .orderBy(F.desc("AvgArrDelay"))
)

origin_state_delay.show(20, truncate=False)

+----------------------------------------------+-----------------------+-------+-----------+
|OriginStateName                               |OriginCityName         |Flights|AvgArrDelay|
+----------------------------------------------+-----------------------+-------+-----------+
|Florida                                       |Vero Beach, FL         |42     |48.32      |
|West Virginia                                 |Morgantown, WV         |764    |34.74      |
|U.S. Pacific Trust Territories and Possessions|Pago Pago, TT          |503    |28.98      |
|Maryland                                      |Hagerstown, MD         |1404   |26.3       |
|California                                    |Stockton, CA           |3604   |24.82      |
|Massachusetts                                 |Hyannis, MA            |1121   |23.62      |
|Delaware                                      |Wilmington, DE         |165    |23.43      |
|North Carolina                                |Concord, NC           

### 2.8 State-wise flight volume (origin & destination)

In [24]:
origin_state_volume = (
    df1.groupBy("OriginStateName")
    .count()
    .orderBy(F.desc("count"))
)

dest_state_volume = (
    df1.groupBy("DestStateName")
    .count()
    .orderBy(F.desc("count"))
)

print("Origin State Flight Volume")
origin_state_volume.show(20)

print("Destination State Flight Volume")
dest_state_volume.show(20)

Origin State Flight Volume
+---------------+-------+
|OriginStateName|  count|
+---------------+-------+
|          Texas|4409615|
|     California|4089241|
|        Florida|3456538|
|       Illinois|2320934|
|        Georgia|2078926|
|       New York|1988710|
| North Carolina|1913405|
|       Colorado|1898418|
|       Virginia|1543198|
|     Washington|1235694|
|        Arizona|1185323|
|         Nevada|1123267|
|   Pennsylvania|1026228|
|       Michigan|1000306|
|      Tennessee| 834680|
|     New Jersey| 782096|
|      Minnesota| 765516|
|  Massachusetts| 758879|
|       Missouri| 708015|
|           Utah| 699784|
+---------------+-------+
only showing top 20 rows

Destination State Flight Volume
+--------------+-------+
| DestStateName|  count|
+--------------+-------+
|         Texas|4409467|
|    California|4089330|
|       Florida|3456663|
|      Illinois|2320696|
|       Georgia|2078780|
|      New York|1988609|
|North Carolina|1913388|
|      Colorado|1898292|
|      Virginia|

### 2.9 Top state-to-state traffic pairs
Highest-volume state pairs, excluding intra-state flights.

In [27]:
from pyspark.sql import functions as F

state_pairs = (
    df1
    .filter(F.col("OriginStateName") != F.col("DestStateName"))
    .groupBy("OriginStateName", "DestStateName")
    .count()
    .orderBy(F.desc("count"))
)

state_pairs.show(20, truncate=False)

+---------------+--------------+------+
|OriginStateName|DestStateName |count |
+---------------+--------------+------+
|Texas          |California    |377502|
|California     |Texas         |377410|
|Florida        |Texas         |367607|
|Texas          |Florida       |367515|
|Florida        |New York      |341623|
|New York       |Florida       |341612|
|Florida        |Georgia       |334181|
|Georgia        |Florida       |334131|
|California     |Nevada        |327390|
|Nevada         |California    |327188|
|Washington     |California    |286369|
|California     |Washington    |286327|
|Arizona        |California    |285721|
|California     |Arizona       |285558|
|Florida        |North Carolina|255147|
|North Carolina |Florida       |255143|
|Colorado       |California    |249916|
|California     |Colorado      |249770|
|Texas          |Colorado      |216875|
|Colorado       |Texas         |216844|
+---------------+--------------+------+
only showing top 20 rows

### 2.10 City-to-city traffic pairs


In [31]:
City_pairs = (
    df1
    .filter(F.col("OriginCityName") != F.col("DestCityName"))
    .groupBy("OriginCityName", "DestCityName")
    .count()
    .orderBy(F.desc("count"))
)

state_pairs.show(20, truncate=False)

+---------------+--------------+------+
|OriginStateName|DestStateName |count |
+---------------+--------------+------+
|Texas          |California    |377502|
|California     |Texas         |377410|
|Florida        |Texas         |367607|
|Texas          |Florida       |367515|
|Florida        |New York      |341623|
|New York       |Florida       |341612|
|Florida        |Georgia       |334181|
|Georgia        |Florida       |334131|
|California     |Nevada        |327390|
|Nevada         |California    |327188|
|Washington     |California    |286369|
|California     |Washington    |286327|
|Arizona        |California    |285721|
|California     |Arizona       |285558|
|Florida        |North Carolina|255147|
|North Carolina |Florida       |255143|
|Colorado       |California    |249916|
|California     |Colorado      |249770|
|Texas          |Colorado      |216875|
|Colorado       |Texas         |216844|
+---------------+--------------+------+
only showing top 20 rows

### 2.11 Arrival delay — standard deviation & skewness

In [22]:
delay_distribution = (
    df1.select(
        F.round(F.stddev("ArrDelay"), 2).alias("StdDeviation"),
        F.round(F.skewness("ArrDelay"), 2).alias("Skewness")
    )
)

delay_distribution.show(truncate=False)

+------------+--------+
|StdDeviation|Skewness|
+------------+--------+
|54.94       |10.58   |
+------------+--------+

### 2.12 Distinct carrier / codeshare values

In [30]:
from pyspark.sql import functions as F

print("Distinct Operating Airlines")

df1.select("Operating_Airline") \
   .distinct() \
   .orderBy("Operating_Airline") \
   .show(100, truncate=False)

print("Count:", df1.select("Operating_Airline").distinct().count())

Distinct Operating Airlines
+-----------------+
|Operating_Airline|
+-----------------+
|9E               |
|AA               |
|AS               |
|AX               |
|B6               |
|C5               |
|CP               |
|DL               |
|EM               |
|EV               |
|F9               |
|G4               |
|G7               |
|HA               |
|MQ               |
|NK               |
|OH               |
|OO               |
|PT               |
|QX               |
|UA               |
|WN               |
|YV               |
|YX               |
|ZW               |
+-----------------+

Count: 25

In [32]:
print("Distinct Marketing Airline Networks")

df1.select("Marketing_Airline_Network") \
   .distinct() \
   .orderBy("Marketing_Airline_Network") \
   .show(100, truncate=False)

print("Count:", df1.select("Marketing_Airline_Network").distinct().count())

Distinct Marketing Airline Networks
+-------------------------+
|Marketing_Airline_Network|
+-------------------------+
|AA                       |
|AS                       |
|B6                       |
|DL                       |
|F9                       |
|G4                       |
|HA                       |
|NK                       |
|UA                       |
|WN                       |
+-------------------------+

Count: 10

In [33]:
print("Distinct Code Share Partner Combinations")

df1.select("Operated_or_Branded_Code_Share_Partners") \
   .distinct() \
   .orderBy("Operated_or_Branded_Code_Share_Partners") \
   .show(200, truncate=False)

print("Count:",
      df1.select("Operated_or_Branded_Code_Share_Partners")
         .distinct()
         .count())

Distinct Code Share Partner Combinations
+---------------------------------------+
|Operated_or_Branded_Code_Share_Partners|
+---------------------------------------+
|AA                                     |
|AA_CODESHARE                           |
|AS                                     |
|AS_CODESHARE                           |
|B6                                     |
|DL                                     |
|DL_CODESHARE                           |
|F9                                     |
|G4                                     |
|HA                                     |
|HA_CODESHARE                           |
|NK                                     |
|UA                                     |
|UA_CODESHARE                           |
|WN                                     |
+---------------------------------------+

Count: 15

### 2.14 Year-over-year delay trend

In [34]:
yoy_delay = (
    df1.groupBy("Year")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay")
    )
    .orderBy("Year")
)

yoy_delay.show(truncate=False)

+----+-------+-----------+-----------+
|Year|Flights|AvgArrDelay|AvgDepDelay|
+----+-------+-----------+-----------+
|2020|5022397|-4.86      |2.06       |
|2021|6311871|3.29       |9.47       |
|2022|7013508|6.96       |12.48      |
|2023|7278739|6.63       |12.21      |
|2024|7546968|7.0        |12.51      |
|2025|7736770|8.53       |13.52      |
+----+-------+-----------+-----------+

### 2.15 On-time vs. delayed vs. cancelled vs. diverted — overall split

In [38]:
from pyspark.sql import functions as F

total = df1.count()

flight_status = (
    df1.withColumn(
        "Status",
        F.when(F.col("Cancelled") == 1, "Cancelled")
         .when(F.col("Diverted") == 1, "Diverted")
         .when(F.col("ArrDel15") == 1, "Delayed")
         .otherwise("On-Time")
    )
    .groupBy("Status")
    .count()
    .withColumn(
        "Percentage",
        F.round(F.col("count") / total * 100, 2)
    )
    .orderBy(
        F.when(F.col("Status") == "On-Time", 1)
         .when(F.col("Status") == "Delayed", 2)
         .when(F.col("Status") == "Cancelled", 3)
         .otherwise(4)
    )
)

flight_status.show(truncate=False)

+---------+--------+----------+
|Status   |count   |Percentage|
+---------+--------+----------+
|On-Time  |32251107|78.83     |
|Delayed  |7644272 |18.69     |
|Cancelled|917084  |2.24      |
|Diverted |97790   |0.24      |
+---------+--------+----------+

### 2.16 Combined state performance dashboard

One table per origin state combining volume, average delays, cancellation rate, and diversion rate — a single view for comparing states head-to-head.

In [37]:
state_dashboard = (
    df1.groupBy("OriginStateName")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum("Cancelled") / F.count("*") * 100, 2).alias("CancellationRate"),
        F.round(F.sum("Diverted") / F.count("*") * 100, 2).alias("DiversionRate")
    )
    .orderBy(F.desc("Flights"))
)

state_dashboard.show(20, truncate=False)

+---------------+-------+-----------+-----------+----------------+-------------+
|OriginStateName|Flights|AvgDepDelay|AvgArrDelay|CancellationRate|DiversionRate|
+---------------+-------+-----------+-----------+----------------+-------------+
|Texas          |4409615|13.07      |7.69       |2.54            |0.24         |
|California     |4089241|8.83       |2.61       |1.81            |0.23         |
|Florida        |3456538|13.86      |8.91       |2.42            |0.27         |
|Illinois       |2320934|12.67      |6.5        |2.5             |0.24         |
|Georgia        |2078926|9.78       |4.66       |1.64            |0.22         |
|New York       |1988710|12.12      |4.06       |3.2             |0.25         |
|North Carolina |1913405|10.6       |5.61       |2.23            |0.23         |
|Colorado       |1898418|14.01      |8.29       |2.23            |0.27         |
|Virginia       |1543198|11.13      |5.27       |2.73            |0.22         |
|Washington     |1235694|6.9

### 2.17 Summary statistics for numeric columns

#### Delay-cause summaries (only rows where that cause is > 0)

In [5]:
c_delay_df = df1.filter(df1.CarrierDelay > 0.0).select('CarrierDelay')
c_delay_df.summary().show()

+-------+-----------------+
|summary|     CarrierDelay|
+-------+-----------------+
|  count|          4256432|
|   mean|  45.372375971236|
| stddev|96.50105571333697|
|    min|              1.0|
|    25%|              9.0|
|    50%|             20.0|
|    75%|             44.0|
|    max|           7232.0|
+-------+-----------------+

In [6]:
w_delay_df = df1.filter(df1.WeatherDelay > 0.0).select('WeatherDelay')
w_delay_df.summary().show()

+-------+------------------+
|summary|      WeatherDelay|
+-------+------------------+
|  count|            462135|
|   mean| 70.78335551299945|
| stddev|120.27591269990215|
|    min|               1.0|
|    25%|              15.0|
|    50%|              34.0|
|    75%|              78.0|
|    max|            2419.0|
+-------+------------------+

In [7]:
n_delay_df = df1.filter(df1.NASDelay > 0.0).select('NASDelay')
n_delay_df.summary().show()

+-------+-----------------+
|summary|         NASDelay|
+-------+-----------------+
|  count|          3703330|
|   mean|27.13804089832664|
| stddev|41.58144075346052|
|    min|              1.0|
|    25%|              8.0|
|    50%|             17.0|
|    75%|             30.0|
|    max|           2700.0|
+-------+-----------------+

In [ ]:
s_delay_df = df1.filter(df1.SecurityDelay > 0.0).select('SecurityDelay')
s_delay_df.summary().show()

+-------+------------------+
|summary|     SecurityDelay|
+-------+------------------+
|  count|             38359|
|   mean|27.109231210406946|
| stddev|41.731644101729955|
|    min|               1.0|
|    25%|              10.0|
|    50%|              18.0|
|    75%|              30.0|
|    max|            1460.0|
+-------+------------------+

In [9]:
l_delay_df = df1.filter(df1.LateAircraftDelay > 0.0).select('LateAircraftDelay')
l_delay_df.summary().show()

+-------+------------------+
|summary| LateAircraftDelay|
+-------+------------------+
|  count|           3738361|
|   mean|55.465691248116485|
| stddev| 77.74762764641166|
|    min|               1.0|
|    25%|              16.0|
|    50%|              32.0|
|    75%|              67.0|
|    max|            3581.0|
+-------+------------------+

#### Overall delay summary (arrival & departure, delayed flights only)

In [10]:
a_delay_df = df1.filter(df1.ArrDelay > 0.0).select('ArrDelay')
a_delay_df.summary().show()

+-------+-----------------+
|summary|         ArrDelay|
+-------+-----------------+
|  count|         13687235|
|   mean|41.89847598875887|
| stddev|  81.224983438403|
|    min|              1.0|
|    25%|              7.0|
|    50%|             18.0|
|    75%|             46.0|
|    max|           7232.0|
+-------+-----------------+

In [11]:
d_delay_df = df1.filter(df1.DepDelay > 0.0).select('DepDelay')
d_delay_df.summary().show()

+-------+------------------+
|summary|          DepDelay|
+-------+------------------+
|  count|          13843115|
|   mean|41.626027956857975|
| stddev|  81.6053026712807|
|    min|               1.0|
|    25%|               6.0|
|    50%|              17.0|
|    75%|              46.0|
|    max|            7223.0|
+-------+------------------+

#### Distance summary

In [12]:
df1.select('Distance').summary().show()

+-------+-----------------+
|summary|         Distance|
+-------+-----------------+
|  count|         40910253|
|   mean|796.3573116010796|
| stddev|585.8496646642392|
|    min|             11.0|
|    25%|            370.0|
|    50%|            642.0|
|    75%|           1034.0|
|    max|           5812.0|
+-------+-----------------+

#### Taxi-in / taxi-out summary

In [15]:
taxiin_df = df1.filter(df1.TaxiIn >= 0.0).select('TaxiIn')
taxiin_df.summary().show()

+-------+-----------------+
|summary|           TaxiIn|
+-------+-----------------+
|  count|         39983837|
|   mean|7.998090303339322|
| stddev|6.609206705002869|
|    min|              0.0|
|    25%|              4.0|
|    50%|              6.0|
|    75%|              9.0|
|    max|           1318.0|
+-------+-----------------+

In [16]:
taxiout_df = df1.filter(df1.TaxiOut >= 0.0).select('TaxiOut')
taxiout_df.summary().show()

+-------+-----------------+
|summary|          TaxiOut|
+-------+-----------------+
|  count|         39997567|
|   mean|17.33446291870703|
| stddev| 9.57135638539504|
|    min|              0.0|
|    25%|             12.0|
|    50%|             15.0|
|    75%|             20.0|
|    max|           1274.0|
+-------+-----------------+

---
# Part 3 — Airport & Airline Reliability, Seasonality, Correlation & Outliers


## Departure vs. Arrival Reliability Split per Airport
Compares, for every airport, its average **departure** delay (as an origin) against its average
**arrival** delay (as a destination) to see whether an airport's delays are mostly self-inflicted
on departure or picked up en route/on arrival.

In [7]:
from pyspark.sql import functions as F

In [8]:
origin_perf = df1.groupBy("Origin").agg(
    F.count("*").alias("Departures"),
    F.avg("DepDelay").alias("AvgDepDelay")
).persist()

origin_perf.count()

387

In [9]:
dest_perf = df1.groupBy("Dest").agg(
    F.count("*").alias("Arrivals"),
    F.avg("ArrDelay").alias("AvgArrDelay")
).persist()

dest_perf.count()

388

In [12]:
airport_perf.cache()
airport_perf.count()

388

Corrected join (broadcast) used to build `airport_perf`:

In [10]:
from pyspark.sql.functions import broadcast

airport_perf = (
    origin_perf.alias("o")
    .join(
        broadcast(dest_perf.alias("d")),
        F.col("o.Origin") == F.col("d.Dest"),
        "outer"
    )
)

### 3.4 Final airport reliability comparison (departures ≥ 1,000 flights)

In [11]:
origin_perf = (
    df1.groupBy("Origin")
    .agg(F.count("*").alias("Departures"), F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"))
)
dest_perf = (
    df1.groupBy("Dest")
    .agg(F.count("*").alias("Arrivals"), F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"))
)

airport_perf = (
    origin_perf.join(dest_perf, origin_perf.Origin == dest_perf.Dest, "outer")
    .withColumn("Airport", F.coalesce(origin_perf.Origin, dest_perf.Dest))
    .select("Airport", "Departures", "AvgDepDelay", "Arrivals", "AvgArrDelay")
    .withColumn("DepMinusArrDelay", F.round(F.col("AvgDepDelay") - F.col("AvgArrDelay"), 2))
    .filter(F.col("Departures") >= 1000)
    .orderBy(F.desc("DepMinusArrDelay"))
)
airport_perf.show(20, truncate=False)   # airports where departing is worse than arriving
airport_perf.orderBy("DepMinusArrDelay").show(20, truncate=False)  # the reverse


+-------+----------+-----------+--------+-----------+----------------+
|Airport|Departures|AvgDepDelay|Arrivals|AvgArrDelay|DepMinusArrDelay|
+-------+----------+-----------+--------+-----------+----------------+
|RFD    |2552      |17.97      |2541    |1.69       |16.28           |
|SCK    |1772      |26.2       |1788    |10.48      |15.72           |
|HDN    |3912      |17.56      |3918    |2.52       |15.04           |
|PSM    |1164      |15.32      |1158    |1.0        |14.32           |
|ACK    |5863      |16.59      |5754    |2.43       |14.16           |
|GRI    |1764      |15.97      |1784    |2.47       |13.5            |
|MVY    |3396      |15.57      |3365    |2.24       |13.33           |
|IAG    |1361      |16.65      |1396    |3.42       |13.23           |
|ORH    |4144      |16.08      |4155    |3.01       |13.07           |
|CKB    |1021      |23.1       |1022    |10.62      |12.48           |
|JAC    |10568     |18.26      |10596   |5.82       |12.44           |
|MTJ  

## Delay-Cause Decomposition per Airport
For airports with at least 1,000 delayed flights, breaks down the *average minutes* attributable
to each delay cause (carrier, weather, NAS/air-traffic-control, security, late-arriving aircraft)
and tags each airport with its single **dominant cause**.

In [19]:
airport_cause = (
    df1
    .filter(F.col("ArrDel15") == 1)
    .groupBy("Origin")
    .agg(
        F.count("*").alias("DelayedFlights"),
        F.round(F.avg("CarrierDelay"), 2).alias("AvgCarrierDelay"),
        F.round(F.avg("WeatherDelay"), 2).alias("AvgWeatherDelay"),
        F.round(F.avg("NASDelay"), 2).alias("AvgNASDelay"),
        F.round(F.avg("SecurityDelay"), 2).alias("AvgSecurityDelay"),
        F.round(F.avg("LateAircraftDelay"), 2).alias("AvgLateAircraftDelay"),
    )
    .filter(F.col("DelayedFlights") >= 1000)
)



In [20]:

# Which cause dominates at each airport?
cause_cols = ["AvgCarrierDelay", "AvgWeatherDelay", "AvgNASDelay", "AvgSecurityDelay", "AvgLateAircraftDelay"]
airport_cause = airport_cause.withColumn(
    "DominantCause",
    F.array_max(F.array(*[F.struct(F.col(c).alias("v"), F.lit(c).alias("k")) for c in cause_cols]))["k"]
)
airport_cause.orderBy(F.desc("DelayedFlights")).show(20, truncate=False)

airport_cause.groupBy("DominantCause").count().orderBy(F.desc("count")).show()

+------+--------------+---------------+---------------+-----------+----------------+--------------------+--------------------+
|Origin|DelayedFlights|AvgCarrierDelay|AvgWeatherDelay|AvgNASDelay|AvgSecurityDelay|AvgLateAircraftDelay|DominantCause       |
+------+--------------+---------------+---------------+-----------+----------------+--------------------+--------------------+
|DFW   |199021        |23.26          |4.74           |10.25      |0.1             |30.7                |AvgLateAircraftDelay|
|ATL   |154434        |27.79          |3.68           |11.85      |0.1             |17.8                |AvgCarrierDelay     |
|ORD   |143559        |21.73          |5.0            |14.12      |0.06            |30.4                |AvgLateAircraftDelay|
|DEN   |138602        |22.14          |2.28           |9.52       |0.09            |26.33               |AvgLateAircraftDelay|
|CLT   |134671        |20.03          |5.93           |10.9       |0.22            |29.4                |AvgLat

## Airline Reliability (On-Time Performance)
Full OTP breakdown per marketing airline network: on-time / delayed / cancelled / diverted
percentages and average arrival delay, ranked by on-time percentage.

In [21]:
airline_otp = (
    df1
    .groupBy("Marketing_Airline_Network")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.sum(F.when(F.col("ArrDel15") == 1, 1).otherwise(0)).alias("DelayedFlights"),
        F.sum(F.when((F.col("ArrDel15") == 0) | (F.col("ArrDel15").isNull() & (F.col("Cancelled") == 0)), 1).otherwise(0)).alias("OnTimeFlights"),
        F.sum("Cancelled").alias("CancelledFlights"),
        F.sum("Diverted").alias("DivertedFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
    )
    .withColumn("OnTimePct", F.round(F.col("OnTimeFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("DelayedPct", F.round(F.col("DelayedFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("CancelPct", F.round(F.col("CancelledFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("DivertPct", F.round(F.col("DivertedFlights") / F.col("TotalFlights") * 100, 2))
    .orderBy(F.desc("OnTimePct"))
)
airline_otp.show(20,vertical = True,truncate=False)

-RECORD 0-----------------------------
 Marketing_Airline_Network | DL       
 TotalFlights              | 3765233  
 DelayedFlights            | 551674   
 OnTimeFlights             | 3149814  
 CancelledFlights          | 63745.0  
 DivertedFlights           | 7434.0   
 AvgArrDelay               | 1.05     
 OnTimePct                 | 83.66    
 DelayedPct                | 14.65    
 CancelPct                 | 1.69     
 DivertPct                 | 0.2      
-RECORD 1-----------------------------
 Marketing_Airline_Network | HA       
 TotalFlights              | 220194   
 DelayedFlights            | 36984    
 OnTimeFlights             | 179568   
 CancelledFlights          | 3642.0   
 DivertedFlights           | 226.0    
 AvgArrDelay               | 5.29     
 OnTimePct                 | 81.55    
 DelayedPct                | 16.8     
 CancelPct                 | 1.65     
 DivertPct                 | 0.1      
-RECORD 2-----------------------------
 Marketing_Airline_Networ

Same airlines, sorted by cancellation rate instead:

In [22]:
airline_otp.select(
    "Marketing_Airline_Network", "TotalFlights", "CancelPct", "DivertPct"
).orderBy(F.desc("CancelPct")).show(20, truncate=False)


+-------------------------+------------+---------+---------+
|Marketing_Airline_Network|TotalFlights|CancelPct|DivertPct|
+-------------------------+------------+---------+---------+
|G4                       |364817      |3.62     |0.31     |
|F9                       |515167      |2.51     |0.15     |
|B6                       |643385      |2.45     |0.36     |
|AS                       |910426      |2.2      |0.24     |
|AA                       |5367138     |2.19     |0.27     |
|UA                       |2031342     |2.01     |0.26     |
|WN                       |4172760     |1.84     |0.22     |
|DL                       |3765233     |1.69     |0.2      |
|HA                       |220194      |1.65     |0.1      |
|NK                       |536981      |1.56     |0.19     |
+-------------------------+------------+---------+---------+

## Airline Reliability at the Busiest Airports
Narrows to the 10 busiest origin airports, then compares each airline's departure-delay
performance specifically at those hubs — first as a long table, then pivoted so each airport is
its own column for quick side-by-side comparison.

In [23]:
top_airports = (
    df1.groupBy("Origin").count().orderBy(F.desc("count")).limit(10)
    .select("Origin").rdd.flatMap(lambda x: x).collect()
)

In [24]:
airline_airport_reliability = (
    df1
    .filter(F.col("Origin").isin(top_airports))
    .groupBy("Marketing_Airline_Network", "Origin")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay"),
        F.round(F.sum(F.when(F.col("DepDel15") == 1, 1).otherwise(0)) / F.count("*") * 100, 2).alias("DelayedPct"),
    )
    .filter(F.col("Flights") >= 500)  # drop thin cells
    .orderBy("Marketing_Airline_Network", F.desc("DelayedPct"))
)
airline_airport_reliability.show(50, truncate=False)

+-------------------------+------+-------+-----------+----------+
|Marketing_Airline_Network|Origin|Flights|AvgDepDelay|DelayedPct|
+-------------------------+------+-------+-----------+----------+
|AA                       |DFW   |774463 |14.89      |23.29     |
|AA                       |ORD   |368414 |13.2       |20.11     |
|AA                       |CLT   |613138 |10.57      |18.64     |
|AA                       |SEA   |20866  |16.57      |18.57     |
|AA                       |DEN   |30244  |15.68      |18.41     |
|AA                       |LAS   |44974  |14.56      |17.69     |
|AA                       |ATL   |39584  |13.48      |17.45     |
|AA                       |LAX   |109315 |12.78      |17.2      |
|AA                       |PHX   |216463 |10.0       |16.87     |
|AA                       |LGA   |136734 |9.65       |16.54     |
|AS                       |DFW   |6047   |12.95      |23.91     |
|AS                       |DEN   |6110   |10.79      |22.29     |
|AS       

Pivoted (airline × top-10-airport) view:

In [25]:
pivot_df = airline_airport_reliability.groupBy("Marketing_Airline_Network").pivot("Origin", top_airports).agg(F.first("DelayedPct"))
pivot_df.show(truncate=False)

+-------------------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Marketing_Airline_Network|ATL  |DFW  |ORD  |CLT  |DEN  |PHX  |LAS  |LGA  |SEA  |LAX  |
+-------------------------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|UA                       |13.7 |15.58|15.62|14.72|17.79|11.79|12.46|14.65|12.15|11.21|
|NK                       |24.7 |21.82|22.55|26.85|19.6 |23.99|18.96|18.23|19.14|18.01|
|AA                       |17.45|23.29|20.11|18.64|18.41|16.87|17.69|16.54|18.57|17.2 |
|B6                       |26.28|27.56|22.7 |20.05|32.89|34.43|27.96|26.48|35.64|20.86|
|DL                       |15.57|16.4 |16.86|14.01|16.14|11.53|13.74|17.04|13.14|14.9 |
|F9                       |35.72|28.98|26.98|30.18|22.85|21.29|24.21|35.8 |28.1 |30.35|
|HA                       |null |null |null |null |null |12.57|21.03|null |24.19|15.88|
|G4                       |null |null |null |null |28.79|29.36|17.75|null |null |19.4 |
|AS                       |20.7 

## Year-over-Year Delay Trend by Airline
Long-form yearly stats per airline, then a pivoted table (rows = airline, columns = year) for the
**top 6 carriers by volume**, showing how each one's delayed-flight percentage has moved 2020–2025.

In [26]:
airline_yearly = (
    df1
    .groupBy("Marketing_Airline_Network", "Year")
    .agg(
        F.count("*").alias("Flights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
        F.round(F.sum(F.when(F.col("ArrDel15") == 1, 1).otherwise(0)) / F.count("*") * 100, 2).alias("DelayedPct"),
    )
    .orderBy("Marketing_Airline_Network", "Year")
)
airline_yearly.show(60, truncate=False)

+-------------------------+----+-------+-----------+----------+
|Marketing_Airline_Network|Year|Flights|AvgArrDelay|DelayedPct|
+-------------------------+----+-------+-----------+----------+
|AA                       |2020|870380 |-1.91      |11.99     |
|AA                       |2021|643551 |2.98       |16.5      |
|AA                       |2022|902739 |9.91       |21.73     |
|AA                       |2023|1046832|7.02       |19.2      |
|AA                       |2024|837003 |8.11       |20.57     |
|AA                       |2025|1066633|13.12      |24.67     |
|AS                       |2020|161980 |-5.47      |10.05     |
|AS                       |2021|79904  |2.14       |18.13     |
|AS                       |2022|137032 |5.46       |21.29     |
|AS                       |2023|167911 |1.84       |17.81     |
|AS                       |2024|167372 |4.64       |20.56     |
|AS                       |2025|196227 |4.82       |21.43     |
|B6                       |2020|105818 |

In [27]:
# Spark-native pivot: one row per airline, one column per year, cell = DelayedPct
top6 = [r[0] for r in df1.groupBy("Marketing_Airline_Network").count().orderBy(F.desc("count")).limit(6).collect()]

yoy_pivot = (
    airline_yearly
    .filter(F.col("Marketing_Airline_Network").isin(top6))
    .groupBy("Marketing_Airline_Network")
    .pivot("Year")
    .agg(F.first("DelayedPct"))
    .orderBy("Marketing_Airline_Network")
)
print("Year-over-Year Delayed-Flight % by Airline (Top 6 Carriers) — pivoted table replaces the line chart:")
yoy_pivot.show(truncate=False)



Year-over-Year Delayed-Flight % by Airline (Top 6 Carriers) — pivoted table replaces the line chart:
+-------------------------+-----+-----+-----+-----+-----+-----+
|Marketing_Airline_Network|2020 |2021 |2022 |2023 |2024 |2025 |
+-------------------------+-----+-----+-----+-----+-----+-----+
|AA                       |11.99|16.5 |21.73|19.2 |20.57|24.67|
|AS                       |10.05|18.13|21.29|17.81|20.56|21.43|
|B6                       |12.35|22.46|32.19|29.42|23.82|25.39|
|DL                       |7.05 |9.91 |16.11|14.13|14.84|19.13|
|UA                       |9.97 |18.15|18.88|13.7 |15.97|23.08|
|WN                       |5.59 |23.75|25.15|20.53|22.6 |22.69|
+-------------------------+-----+-----+-----+-----+-----+-----+

---
## Delay Analysis by Season & Day of Week
A `Season` column is derived from `Month` (Winter/Spring/Summer/Fall), then average delay is
pivoted by day of week × season — once for arrival delay, once for departure delay.

**Average Arrival Delay: Day of Week × Season**

In [28]:
from pyspark.sql import functions as F

# Create Season column based on Month
df1_season = df1.withColumn(
    "Season",
    F.when(F.col("Month").isin(12, 1, 2), "Winter")
     .when(F.col("Month").isin(3, 4, 5), "Spring")
     .when(F.col("Month").isin(6, 7, 8), "Summer")
     .otherwise("Fall")
)







In [30]:
# Average Arrival Delay by Day of Week and Season
dow_season = (
    df1_season
    .groupBy("DayOfWeek", "Season")
    .agg(
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay")
    )
)

In [31]:
# Pivot the seasons into columns
dow_season_pivot = (
    dow_season
    .groupBy("DayOfWeek")
    .pivot("Season", ["Winter", "Spring", "Summer", "Fall"])
    .agg(F.first("AvgArrDelay"))
    .orderBy("DayOfWeek")
)

In [32]:
print("Average Arrival Delay: Day of Week × Season")
dow_season_pivot.show(truncate=False)

Average Arrival Delay: Day of Week × Season
+---------+------+------+------+-----+
|DayOfWeek|Winter|Spring|Summer|Fall |
+---------+------+------+------+-----+
|1        |4.38  |3.54  |10.23 |1.32 |
|2        |2.54  |-0.09 |6.5   |-2.23|
|3        |2.58  |2.67  |7.55  |-2.1 |
|4        |4.83  |6.22  |11.51 |1.91 |
|5        |4.39  |7.23  |11.2  |2.67 |
|6        |3.73  |5.23  |8.08  |-1.07|
|7        |5.75  |5.18  |11.96 |3.17 |
+---------+------+------+------+-----+

**Average Departure Delay: Day of Week × Season**

In [33]:
# Average Departure Delay by Day of Week and Season
dow_season = (
    df1_season
    .groupBy("DayOfWeek", "Season")
    .agg(
        F.round(F.avg("DepDelay"), 2).alias("AvgDepDelay")
    )
)

In [34]:
# Pivot the seasons into columns
dow_season_pivot = (
    dow_season
    .groupBy("DayOfWeek")
    .pivot("Season", ["Winter", "Spring", "Summer", "Fall"])
    .agg(F.first("AvgDepDelay"))
    .orderBy("DayOfWeek")
)

In [35]:
print("Average Departure Delay: Day of Week × Season")
dow_season_pivot.show(truncate=False)

Average Departure Delay: Day of Week × Season
+---------+------+------+------+----+
|DayOfWeek|Winter|Spring|Summer|Fall|
+---------+------+------+------+----+
|1        |10.58 |9.47  |15.31 |7.44|
|2        |8.99  |6.36  |12.05 |4.48|
|3        |8.74  |8.23  |12.59 |4.42|
|4        |10.48 |11.03 |15.88 |7.54|
|5        |10.44 |12.27 |15.7  |8.61|
|6        |10.5  |11.34 |13.9  |6.09|
|7        |11.97 |11.22 |17.0  |9.02|
+---------+------+------+------+----+

---
## Correlation Between Numeric Operational Fields
Builds a feature vector of delay/operational columns and computes a full Pearson correlation
matrix, useful for spotting redundant features before any downstream modeling.

In [36]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

numeric_cols = [
    "DepDelay", "ArrDelay", "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay", "Distance", "AirTime", "TaxiOut", "TaxiIn",
]

In [37]:
corr_input = df1.select(*numeric_cols).dropna()

assembler = VectorAssembler(inputCols=numeric_cols, outputCol="features")
vector_df = assembler.transform(corr_input).select("features")

corr_matrix = Correlation.corr(vector_df, "features", method="pearson").collect()[0][0]
corr_array = corr_matrix.toArray()

In [38]:
corr_rows = [
    tuple([numeric_cols[i]] + [float(round(x, 3)) for x in corr_array[i]])
    for i in range(len(numeric_cols))
]
corr_spark_df = spark.createDataFrame(corr_rows, ["Feature"] + numeric_cols)
corr_spark_df.show(truncate=False)


----------------------------------------
Exception happened during processing of request from ('127.0.0.1', 53974)
----------------------------------------
+-----------------+--------+--------+------------+------------+--------+-------------+-----------------+--------+-------+-------+------+
|Feature          |DepDelay|ArrDelay|CarrierDelay|WeatherDelay|NASDelay|SecurityDelay|LateAircraftDelay|Distance|AirTime|TaxiOut|TaxiIn|
+-----------------+--------+--------+------------+------------+--------+-------------+-----------------+--------+-------+-------+------+
|DepDelay         |1.0     |0.979   |0.686       |0.269       |0.097   |0.017        |0.566            |-0.01   |-0.032 |-0.113 |-0.069|
|ArrDelay         |0.979   |1.0     |0.683       |0.279       |0.196   |0.015        |0.547            |-0.008  |-0.012 |0.021  |0.019 |
|CarrierDelay     |0.686   |0.683   |1.0         |-0.034      |-0.083  |-0.011       |-0.044           |0.024   |0.012  |-0.054 |-0.035|
|WeatherDelay     |0.2

---
## Candidate Target Variables for Modeling
Explores four possible prediction targets and their class/value distributions: binary arrival
delay (`ArrDel15`), binary cancellation, continuous arrival delay (regression), and binary
departure delay (`DepDel15`) / continuous departure delay.

In [39]:
print("Target option A — Binary classification: ArrDel15 (arrival delayed 15+ min)")
df1.groupBy("ArrDel15").count().withColumn(
    "Pct", F.round(F.col("count") / df1.count() * 100, 2)
).show()

Target option A — Binary classification: ArrDel15 (arrival delayed 15+ min)
+--------+--------+-----+
|ArrDel15|   count|  Pct|
+--------+--------+-----+
|     0.0|14651196|79.08|
|    null|  416418| 2.25|
|     1.0| 3459829|18.67|
+--------+--------+-----+

In [40]:
print("Target option B — Binary classification: Cancelled")
df1.groupBy("Cancelled").count().withColumn(
    "Pct", F.round(F.col("count") / df1.count() * 100, 2)
).show()

Target option B — Binary classification: Cancelled
+---------+--------+-----+
|Cancelled|   count|  Pct|
+---------+--------+-----+
|      0.0|18154858|97.99|
|      1.0|  372585| 2.01|
+---------+--------+-----+

In [41]:
print("Target option C — Regression: ArrDelay (continuous, distribution below)")
df1.select("ArrDelay").describe().show()

Target option C — Regression: ArrDelay (continuous, distribution below)
+-------+-----------------+
|summary|         ArrDelay|
+-------+-----------------+
|  count|         18111025|
|   mean|4.890034329917826|
| stddev|53.86789162608446|
|    min|           -139.0|
|    max|           5986.0|
+-------+-----------------+

In [42]:
from pyspark.sql import functions as F

# Option A — Binary classification: DepDel15
print("Target option A — Binary classification: DepDel15 (departure delayed 15+ min)")

df1.groupBy("DepDel15").count() \
    .withColumn(
        "Pct",
        F.round(F.col("count") / df1.count() * 100, 2)
    ).show()

Target option A — Binary classification: DepDel15 (departure delayed 15+ min)
+--------+--------+-----+
|DepDel15|   count|  Pct|
+--------+--------+-----+
|     0.0|14714671|79.42|
|    null|  363075| 1.96|
|     1.0| 3449697|18.62|
+--------+--------+-----+

In [43]:
# Option B — Regression: DepDelay
print("Target option B — Regression: DepDelay (continuous, distribution below)")

df1.select("DepDelay").describe().show()

Target option B — Regression: DepDelay (continuous, distribution below)
+-------+------------------+
|summary|          DepDelay|
+-------+------------------+
|  count|          18164368|
|   mean|10.636345068542985|
| stddev|51.917928851262765|
|    min|            -131.0|
|    max|            5995.0|
+-------+------------------+

---
## Operating Airline Reliability
Same OTP-style breakdown as before, but grouped by the **operating** airline instead of the
marketing airline network — surfaces regional carriers that fly under a major airline's brand.

> **As-run note:** the `.show()` call in this cell was left pointing at `airline_otp` (marketing-airline table) rather than the newly computed `airline_otp1` (operating-airline table), so the displayed output repeats the marketing-airline OTP table from above. The correct operating-airline table is shown in the very next cell, sorted by cancellation rate. Reproduced as originally run.

In [47]:
airline_otp1 = (
    df1
    .groupBy("Operating_Airline")
    .agg(
        F.count("*").alias("TotalFlights"),
        F.sum(F.when(F.col("ArrDel15") == 1, 1).otherwise(0)).alias("DelayedFlights"),
        F.sum(F.when((F.col("ArrDel15") == 0) | (F.col("ArrDel15").isNull() & (F.col("Cancelled") == 0)), 1).otherwise(0)).alias("OnTimeFlights"),
        F.sum("Cancelled").alias("CancelledFlights"),
        F.sum("Diverted").alias("DivertedFlights"),
        F.round(F.avg("ArrDelay"), 2).alias("AvgArrDelay"),
    )
    .withColumn("OnTimePct", F.round(F.col("OnTimeFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("DelayedPct", F.round(F.col("DelayedFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("CancelPct", F.round(F.col("CancelledFlights") / F.col("TotalFlights") * 100, 2))
    .withColumn("DivertPct", F.round(F.col("DivertedFlights") / F.col("TotalFlights") * 100, 2))
    .orderBy(F.desc("OnTimePct"))
)
airline_otp.show(20,vertical = True,truncate=False)

-RECORD 0-----------------------------
 Marketing_Airline_Network | DL       
 TotalFlights              | 3765233  
 DelayedFlights            | 551674   
 OnTimeFlights             | 3149814  
 CancelledFlights          | 63745.0  
 DivertedFlights           | 7434.0   
 AvgArrDelay               | 1.05     
 OnTimePct                 | 83.66    
 DelayedPct                | 14.65    
 CancelPct                 | 1.69     
 DivertPct                 | 0.2      
-RECORD 1-----------------------------
 Marketing_Airline_Network | HA       
 TotalFlights              | 220194   
 DelayedFlights            | 36984    
 OnTimeFlights             | 179568   
 CancelledFlights          | 3642.0   
 DivertedFlights           | 226.0    
 AvgArrDelay               | 5.29     
 OnTimePct                 | 81.55    
 DelayedPct                | 16.8     
 CancelPct                 | 1.65     
 DivertPct                 | 0.1      
-RECORD 2-----------------------------
 Marketing_Airline_Networ

Operating airlines sorted by cancellation rate:

In [48]:
airline_otp1.select(
    "Operating_Airline", "TotalFlights", "CancelPct", "DivertPct"
).orderBy(F.desc("CancelPct")).show(20, truncate=False)

+-----------------+------------+---------+---------+
|Operating_Airline|TotalFlights|CancelPct|DivertPct|
+-----------------+------------+---------+---------+
|EM               |4185        |7.91     |0.45     |
|EV               |33669       |6.96     |0.19     |
|CP               |10034       |6.94     |0.15     |
|G4               |364817      |3.62     |0.31     |
|AX               |10191       |3.3      |0.25     |
|YX               |1197358     |3.21     |0.22     |
|YV               |339144      |3.2      |0.26     |
|G7               |130225      |3.12     |0.23     |
|F9               |515167      |2.51     |0.15     |
|B6               |643385      |2.45     |0.36     |
|9E               |751723      |2.44     |0.18     |
|QX               |167834      |2.4      |0.14     |
|AS               |687021      |2.26     |0.28     |
|ZW               |156948      |2.12     |0.2      |
|MQ               |787529      |2.1      |0.21     |
|AA               |2990742     |2.1      |0.27

---
## Marketing vs. Operating Airline — Same-Carrier Flights
Checks how often the marketing and operating airline are literally the same code (i.e. no
codeshare/regional-partner relationship involved).

In [49]:
from pyspark.sql import functions as F

same_airline = (
    df1.filter(
        F.col("Operating_Airline") == F.col("Marketing_Airline_Network")
    )
)

same_airline.select(
    "Marketing_Airline_Network",
    "Operating_Airline"
).distinct().show(truncate=False)

+-------------------------+-----------------+
|Marketing_Airline_Network|Operating_Airline|
+-------------------------+-----------------+
|WN                       |WN               |
|NK                       |NK               |
|F9                       |F9               |
|AS                       |AS               |
|AA                       |AA               |
|B6                       |B6               |
|G4                       |G4               |
|HA                       |HA               |
|UA                       |UA               |
|DL                       |DL               |
+-------------------------+-----------------+

In [50]:
same_count = (
    df1.filter(
        F.col("Operating_Airline") == F.col("Marketing_Airline_Network")
    )
    .count()
)

print("Flights with same Marketing and Operating Airline:", same_count)

('Flights with same Marketing and Operating Airline:', 13391811)

---
## Outlier Detection (IQR Method)
For each key numeric column, computes the interquartile range (Q1–Q3) and counts how many rows
fall outside 1.5×IQR of the box — a standard, non-parametric way to flag extreme values.

In [ ]:
numeric_cols = [
    "ArrDelay",
    "DepDelay",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
    "AirTime",
    "Distance",
    "TaxiOut",
    "TaxiIn"
]

**Arrival Delay**

In [53]:
from pyspark.sql import functions as F

Q1, Q3 = df1.approxQuantile("ArrDelay", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("ArrDelay") < (Q1 - 1.5 * IQR)) |
    (F.col("ArrDelay") > (Q3 + 1.5 * IQR))
).count()

1648905

**Departure Delay**

In [54]:
Q1, Q3 = df1.approxQuantile("DepDelay", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("DepDelay") < (Q1 - 1.5 * IQR)) |
    (F.col("DepDelay") > (Q3 + 1.5 * IQR))
).count()

2233485

**Air Time**

In [55]:
Q1, Q3 = df1.approxQuantile("AirTime", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("AirTime") < (Q1 - 1.5 * IQR)) |
    (F.col("AirTime") > (Q3 + 1.5 * IQR))
).count()

881175

**Distance**

In [56]:
Q1, Q3 = df1.approxQuantile("Distance", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("Distance") < (Q1 - 1.5 * IQR)) |
    (F.col("Distance") > (Q3 + 1.5 * IQR))
).count()

1033569

**Taxi-Out**

In [57]:
Q1, Q3 = df1.approxQuantile("TaxiOut", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("TaxiOut") < (Q1 - 1.5 * IQR)) |
    (F.col("TaxiOut") > (Q3 + 1.5 * IQR))
).count()

988324

**Taxi-In**

In [58]:
Q1, Q3 = df1.approxQuantile("TaxiIn", [0.25, 0.75], 0.01)
IQR = Q3 - Q1

df1.filter(
    (F.col("TaxiIn") < (Q1 - 1.5 * IQR)) |
    (F.col("TaxiIn") > (Q3 + 1.5 * IQR))
).count()

822545

---
## Summary

This combined notebook walks through the full arc of an airline on-time-performance EDA:

1. **Data quality** — no duplicate flight keys, nulls concentrated where expected (cancelled/diverted
   flights lack delay values), sane minimum values.
2. **Overall performance** — 97.76% completion rate, 78.83% on-time, with average delays trending
   up year-over-year (2020 → 2025).
3. **Where delays happen** — concentrated at major hub airports, dominated by *late aircraft* and
   *carrier* causes rather than weather; short-haul routes lose proportionally more time per mile
   than long-haul.
4. **Who is most/least reliable** — airline- and airport-level OTP leaderboards, plus
   airline × airport and airline × year breakdowns for the busiest hubs and top carriers.
5. **When delays happen** — clear seasonal and day-of-week patterns (summer/holiday months worse
   across nearly every carrier).
6. **Modeling groundwork** — a correlation matrix of numeric features, four candidate target
   variables (arrival/departure delay as classification or regression, cancellation), and IQR-based
   outlier counts for every key numeric column, all ready to feed into a downstream ML pipeline.
